In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt


In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")
SCORES_DATA_DIR = Path("../data/scores")
OUTPUTS_DATA_DIR = Path("../data/outputs")

In [3]:
workforce_score_df = pd.read_csv(
    SCORES_DATA_DIR / "workforce_scored_features.csv",
    sep=",",
    encoding="utf-8"
)

actions_score_df = pd.read_csv(
    SCORES_DATA_DIR / "actions_scored_features.csv",
    sep=",",
    encoding="utf-8"
)

commercial_score_df = pd.read_csv(
    SCORES_DATA_DIR / "commercial_scored_features.csv",
    sep=",",
    encoding="utf-8"
)


In [4]:
merge_columns = [
    "course_id",
    "course_name",
    "course_area",
    "local"
]

hiring_analysis_full_df = (
    workforce_score_df
    .merge(
        actions_score_df,
        how="inner",
        on=merge_columns,
        validate="one_to_one"
    )
    .merge(
        commercial_score_df,
        how="inner",
        on=merge_columns,
        validate="one_to_one"
    )
)

hiring_analysis_full_df


,course_id,course_name,course_area,local,trainer_count,pct_multicourse_trainers,pct_mobile_trainers,relative_scarcity_score,relative_scarcity_contribution,course_sharing_contribution,...,commercial_regularity_score,commercial_growth_score,commercial_volume_contribution,commercial_conversion_contribution,commercial_regularity_contribution,commercial_growth_contribution,commercial_base_score,commercial_pressure_tier,commercial_main_pressure_driver,commercial_priority_rank
0,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 3,2,100.0,50.0,86.7,52.0,20.0,...,30.67,30.17,6.08,10.31,7.67,4.53,28.59,Lower-middle,Conversion,99
1,TUR,Atendimento Turístico,Turismo,Centro 4,2,100.0,50.0,86.7,52.0,20.0,...,20.87,29.17,7.26,20.15,5.22,4.38,37.00,Lower-middle,Conversion,82
2,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 2,2,100.0,50.0,86.7,52.0,20.0,...,7.17,48.00,1.40,20.56,1.79,7.20,30.95,Lower-middle,Conversion,90
3,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,2,100.0,50.0,86.7,52.0,20.0,...,64.00,31.67,28.47,19.54,16.00,4.75,68.76,Upper-middle,Volume,17
4,ING,Inglês Profissional,Línguas,Centro 4,2,100.0,50.0,86.7,52.0,20.0,...,50.67,0.00,14.38,12.83,12.67,0.00,39.88,Lower-middle,Volume,77
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 5,4,75.0,0.0,24.6,14.8,15.0,...,27.17,29.17,9.45,10.00,6.79,4.38,30.62,Lower-middle,Conversion,91
116,ING,Inglês Profissional,Línguas,Centro 2,4,75.0,0.0,24.6,14.8,15.0,...,46.79,0.00,26.78,13.84,11.70,0.00,52.32,Upper-middle,Volume,49
117,HAC,Higiene e Segurança Alimentar,Alimentar,Centro 2,5,100.0,40.0,1.7,1.0,20.0,...,27.83,16.92,9.26,23.69,6.96,2.54,42.44,Lower-middle,Conversion,69
118,HAC,Higiene e Segurança Alimentar,Alimentar,Centro 5,4,50.0,0.0,24.6,14.8,10.0,...,79.50,51.38,28.98,21.76,19.88,7.71,78.32,High,Volume,7


## Weighted hiring pressure score

This score combines the three previously developed analytical scores.  
The main driver is the highest original score before weighting.


In [5]:
score_weights = {
    "workforce_base_score": 0.40,
    "actions_base_score": 0.40,
    "commercial_base_score": 0.20
}

base_score_columns = list(score_weights)

contribution_columns = [
    "workforce_contribution",
    "actions_contribution",
    "commercial_contribution"
]

assert abs(sum(score_weights.values()) - 1) < 1e-9, (
    "Hiring score weights must sum to 1."
)


In [6]:
scores_df = (
    hiring_analysis_full_df[
        [
            "course_id",
            "course_name",
            "course_area",
            "local",

            "workforce_base_score",
            "workforce_base_priority_rank",
            "workforce_vulnerability_tier",

            "actions_base_score",
            "actions_priority_rank",
            "actions_pressure_tier",

            "commercial_base_score",
            "commercial_priority_rank",
            "commercial_pressure_tier"
        ]
    ]
    .copy()
    .assign(
        workforce_contribution=lambda df:
            df["workforce_base_score"]
            .mul(score_weights["workforce_base_score"])
            .round(2),

        actions_contribution=lambda df:
            df["actions_base_score"]
            .mul(score_weights["actions_base_score"])
            .round(2),

        commercial_contribution=lambda df:
            df["commercial_base_score"]
            .mul(score_weights["commercial_base_score"])
            .round(2)
    )
)


In [7]:
scores_df = (
    scores_df
    .assign(
        hiring_pressure_score=lambda df:
            df[contribution_columns]
            .sum(axis=1)
            .round(2),

        hiring_pressure_rank=lambda df:
            df["hiring_pressure_score"]
            .rank(
                method="min",
                ascending=False
            )
            .astype(int),

        hiring_main_pressure_driver=lambda df:
            df[base_score_columns]
            .idxmax(axis=1)
            .str.replace("_base_score", "", regex=False)
            .str.title()
    )
)

scores_df["hiring_pressure_tier"] = pd.cut(
    scores_df["hiring_pressure_score"],
    bins=[0, 25, 50, 75, 100],
    labels=[
        "Low",
        "Lower-middle",
        "Upper-middle",
        "High"
    ],
    include_lowest=True
)

scores_df


,course_id,course_name,course_area,local,workforce_base_score,workforce_base_priority_rank,workforce_vulnerability_tier,actions_base_score,actions_priority_rank,actions_pressure_tier,commercial_base_score,commercial_priority_rank,commercial_pressure_tier,workforce_contribution,actions_contribution,commercial_contribution,hiring_pressure_score,hiring_pressure_rank,hiring_main_pressure_driver,hiring_pressure_tier
0,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 3,82.0,1,High,26.14,107,Lower-middle,28.59,99,Lower-middle,32.80,10.46,5.72,48.98,59,Workforce,Lower-middle
1,TUR,Atendimento Turístico,Turismo,Centro 4,82.0,1,High,20.18,111,Low,37.00,82,Lower-middle,32.80,8.07,7.40,48.27,61,Workforce,Lower-middle
2,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 2,82.0,1,High,9.24,118,Low,30.95,90,Lower-middle,32.80,3.70,6.19,42.69,90,Workforce,Lower-middle
3,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,82.0,1,High,76.69,14,High,68.76,17,Upper-middle,32.80,30.68,13.75,77.23,1,Workforce,High
4,ING,Inglês Profissional,Línguas,Centro 4,82.0,1,High,33.40,94,Lower-middle,39.88,77,Lower-middle,32.80,13.36,7.98,54.14,46,Workforce,Upper-middle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 5,29.8,15,Lower-middle,67.23,28,Upper-middle,30.62,91,Lower-middle,11.92,26.89,6.12,44.93,78,Actions,Lower-middle
116,ING,Inglês Profissional,Línguas,Centro 2,29.8,15,Lower-middle,61.78,36,Upper-middle,52.32,49,Upper-middle,11.92,24.71,10.46,47.09,68,Actions,Lower-middle
117,HAC,Higiene e Segurança Alimentar,Alimentar,Centro 2,29.0,16,Lower-middle,40.80,77,Lower-middle,42.44,69,Lower-middle,11.60,16.32,8.49,36.41,101,Commercial,Lower-middle
118,HAC,Higiene e Segurança Alimentar,Alimentar,Centro 5,24.8,17,Low,68.24,26,Upper-middle,78.32,7,High,9.92,27.30,15.66,52.88,51,Commercial,Upper-middle


## Crossed hiring pressure score

This complementary score compares operational and commercial demand with the current trainer capacity.


In [8]:
time_weights = {
    "recent": 0.60,
    "historical": 0.40
}

cross_score_weights = {
    "workload_pressure_score": 0.30,
    "concurrency_pressure_score": 0.25,
    "commercial_coverage_pressure_score": 0.15,
    "sustained_demand_pressure_score": 0.15,
    "shared_trainer_capacity_pressure_score": 0.15
}

cross_driver_labels = {
    "workload_pressure_score": "Training workload",
    "concurrency_pressure_score": "Concurrent actions",
    "commercial_coverage_pressure_score": "Commercial demand",
    "sustained_demand_pressure_score": "Sustained demand",
    "shared_trainer_capacity_pressure_score": "Shared trainer capacity"
}

assert abs(sum(time_weights.values()) - 1) < 1e-9, (
    "Time weights must sum to 1."
)

assert abs(sum(cross_score_weights.values()) - 1) < 1e-9, (
    "Cross score weights must sum to 1."
)

In [9]:
cross_scores_df = (
    hiring_analysis_full_df[
        merge_columns
        + [
            "trainer_count",
            "pct_mobile_trainers",
            "pct_multicourse_trainers",

            "training_hours_monthly_avg_last_12m",
            "training_hours_monthly_avg_historical",

            "peak_concurrent_actions_last_12m",
            "peak_concurrent_actions_historical",

            "active_month_ratio_last_12m",
            "active_month_ratio_historical",

            "pct_simultaneous_action_days_last_12m",
            "pct_simultaneous_action_days_historical",

            "avg_sales_last_13w",
            "avg_sales_historical",

            "pct_with_sales_last_13w",
            "pct_with_sales_historical"
        ]
    ]
    .copy()
)


In [10]:
cross_scores_df = (
    cross_scores_df
    .assign(
        weighted_training_hours=lambda df:
            df["training_hours_monthly_avg_last_12m"]
            .mul(time_weights["recent"])
            .add(
                df["training_hours_monthly_avg_historical"]
                .mul(time_weights["historical"])
            ),

        weighted_concurrent_actions=lambda df:
            df["peak_concurrent_actions_last_12m"]
            .mul(time_weights["recent"])
            .add(
                df["peak_concurrent_actions_historical"]
                .mul(time_weights["historical"])
            ),

        weighted_weekly_sales=lambda df:
            df["avg_sales_last_13w"]
            .mul(time_weights["recent"])
            .add(
                df["avg_sales_historical"]
                .mul(time_weights["historical"])
            ),

        weighted_active_month_ratio=lambda df:
            df["active_month_ratio_last_12m"]
            .mul(time_weights["recent"])
            .add(
                df["active_month_ratio_historical"]
                .mul(time_weights["historical"])
            ),

        weighted_pct_with_sales=lambda df:
            df["pct_with_sales_last_13w"]
            .mul(time_weights["recent"])
            .add(
                df["pct_with_sales_historical"]
                .mul(time_weights["historical"])
            ),

        weighted_simultaneous_days=lambda df:
            df["pct_simultaneous_action_days_last_12m"]
            .mul(time_weights["recent"])
            .add(
                df["pct_simultaneous_action_days_historical"]
                .mul(time_weights["historical"])
            ),

        shared_trainer_ratio=lambda df:
            df[
                [
                    "pct_mobile_trainers",
                    "pct_multicourse_trainers"
                ]
            ]
            .mean(axis=1)
            .div(100)
    )
)


In [11]:
cross_scores_df = (
    cross_scores_df
    .assign(
        training_hours_per_trainer=lambda df:
            df["weighted_training_hours"]
            .div(df["trainer_count"]),

        concurrent_actions_per_trainer=lambda df:
            df["weighted_concurrent_actions"]
            .div(df["trainer_count"]),

        weekly_sales_per_trainer=lambda df:
            df["weighted_weekly_sales"]
            .div(df["trainer_count"]),

        sustained_demand_pressure=lambda df:
            df["weighted_active_month_ratio"]
            .mul(
                df["weighted_pct_with_sales"]
                .div(100)
            ),

        shared_trainer_capacity_pressure=lambda df:
            df["weighted_simultaneous_days"]
            .div(100)
            .mul(df["shared_trainer_ratio"])
    )
)


In [12]:
cross_scores_df = (
    cross_scores_df
    .assign(
        workload_pressure_score=lambda df:
            df["training_hours_per_trainer"]
            .rank(method="average", pct=True)
            .mul(100),

        concurrency_pressure_score=lambda df:
            df["concurrent_actions_per_trainer"]
            .rank(method="average", pct=True)
            .mul(100),

        commercial_coverage_pressure_score=lambda df:
            df["weekly_sales_per_trainer"]
            .rank(method="average", pct=True)
            .mul(100),

        sustained_demand_pressure_score=lambda df:
            df["sustained_demand_pressure"]
            .rank(method="average", pct=True)
            .mul(100),

        shared_trainer_capacity_pressure_score=lambda df:
            df["shared_trainer_capacity_pressure"]
            .rank(method="average", pct=True)
            .mul(100)
    )
)

In [13]:
cross_scores_df = (
    cross_scores_df
    .assign(
        cross_hiring_index=lambda df:
            df[list(cross_score_weights)]
            .mul(pd.Series(cross_score_weights))
            .sum(axis=1),

        cross_main_pressure_driver=lambda df:
            df[list(cross_score_weights)]
            .idxmax(axis=1)
            .map(cross_driver_labels)
    )
)

In [14]:
cross_scores_df = (
    cross_scores_df
    .assign(
        cross_hiring_score=lambda df:
            df["cross_hiring_index"]
            .round(2),
    
        cross_hiring_rank=lambda df:
            df["cross_hiring_score"]
            .rank(
                method="min",
                ascending=False
            )
            .astype(int)
    )
)

In [15]:
cross_scores_df["cross_hiring_tier"] = pd.cut(
    cross_scores_df["cross_hiring_score"],
    bins=[0, 25, 50, 75, 100],
    labels=[
        "Low",
        "Lower-middle",
        "Upper-middle",
        "High"
    ],
    include_lowest=True
)

cross_scores_df


,course_id,course_name,course_area,local,trainer_count,pct_mobile_trainers,pct_multicourse_trainers,training_hours_monthly_avg_last_12m,training_hours_monthly_avg_historical,peak_concurrent_actions_last_12m,...,workload_pressure_score,concurrency_pressure_score,commercial_coverage_pressure_score,sustained_demand_pressure_score,shared_trainer_capacity_pressure_score,cross_hiring_index,cross_main_pressure_driver,cross_hiring_score,cross_hiring_rank,cross_hiring_tier
0,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 3,2,50.0,100.0,35.42,19.72,2,...,68.333333,47.916667,50.000000,20.833333,39.166667,48.979167,Training workload,48.98,60,Lower-middle
1,TUR,Atendimento Turístico,Turismo,Centro 4,2,50.0,100.0,22.67,11.20,2,...,45.833333,47.916667,55.000000,13.333333,36.666667,41.479167,Commercial demand,41.48,75,Lower-middle
2,EMP,Empilhadores e Movimentação de Cargas,Segurança,Centro 2,2,50.0,100.0,18.67,10.67,2,...,33.333333,47.916667,16.666667,4.166667,12.500000,26.979167,Concurrent actions,26.98,97,Lower-middle
3,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,2,50.0,100.0,48.00,32.36,3,...,87.500000,100.000000,96.666667,78.333333,79.166667,89.375000,Concurrent actions,89.38,5,High
4,ING,Inglês Profissional,Línguas,Centro 4,2,50.0,100.0,40.00,26.00,2,...,78.333333,69.166667,71.666667,42.500000,34.583333,63.104167,Training workload,63.10,40,Upper-middle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 5,4,0.0,75.0,120.83,65.00,5,...,90.000000,73.750000,6.666667,17.500000,52.500000,56.937500,Training workload,56.94,49,Upper-middle
116,ING,Inglês Profissional,Línguas,Centro 2,4,0.0,75.0,77.50,31.00,3,...,69.166667,34.583333,61.666667,48.333333,73.333333,56.895833,Shared trainer capacity,56.90,50,Upper-middle
117,HAC,Higiene e Segurança Alimentar,Alimentar,Centro 2,5,40.0,100.0,12.67,9.69,3,...,0.833333,3.333333,8.333333,19.166667,50.833333,12.833333,Shared trainer capacity,12.83,118,Low
118,HAC,Higiene e Segurança Alimentar,Alimentar,Centro 5,4,0.0,50.0,26.67,13.96,3,...,15.833333,23.333333,67.500000,83.333333,20.000000,36.208333,Sustained demand,36.21,84,Lower-middle


In [16]:
scores_df = (
    scores_df
    .merge(
        cross_scores_df[
            merge_columns
            + [
                "training_hours_per_trainer",
                "concurrent_actions_per_trainer",
                "weekly_sales_per_trainer",
                "sustained_demand_pressure",
                "shared_trainer_capacity_pressure",

                "workload_pressure_score",
                "concurrency_pressure_score",
                "commercial_coverage_pressure_score",
                "sustained_demand_pressure_score",
                "shared_trainer_capacity_pressure_score",

                "cross_hiring_rank",
                "cross_hiring_score",
                "cross_hiring_tier",
                "cross_main_pressure_driver"
            ]
        ],
        how="left",
        on=merge_columns,
        validate="one_to_one"
    )
    .assign(
        training_hours_per_trainer=lambda df:
            df["training_hours_per_trainer"].round(2),

        concurrent_actions_per_trainer=lambda df:
            df["concurrent_actions_per_trainer"].round(2),

        weekly_sales_per_trainer=lambda df:
            df["weekly_sales_per_trainer"].round(2),

        sustained_demand_pressure=lambda df:
            df["sustained_demand_pressure"].round(4),

        shared_trainer_capacity_pressure=lambda df:
            df["shared_trainer_capacity_pressure"].round(4)
    )
    .sort_values(
        [
            "hiring_pressure_score",
            "cross_hiring_score"
        ],
        ascending=[False, False],
        ignore_index=True
    )
)


In [17]:
scores_df = scores_df[
    [
        "course_id",
        "course_name",
        "course_area",
        "local",

        "hiring_pressure_rank",
        "hiring_pressure_score",
        "hiring_pressure_tier",
        "hiring_main_pressure_driver",

        "cross_hiring_rank",
        "cross_hiring_score",
        "cross_hiring_tier",
        "cross_main_pressure_driver",

        "training_hours_per_trainer",
        "concurrent_actions_per_trainer",
        "weekly_sales_per_trainer",
        "sustained_demand_pressure",
        "shared_trainer_capacity_pressure",

        "workload_pressure_score",
        "concurrency_pressure_score",
        "commercial_coverage_pressure_score",
        "sustained_demand_pressure_score",
        "shared_trainer_capacity_pressure_score",

        "workforce_base_score",
        "actions_base_score",
        "commercial_base_score",

        "workforce_vulnerability_tier",
        "actions_pressure_tier",
        "commercial_pressure_tier"
    ]
]

scores_df


,course_id,course_name,course_area,local,hiring_pressure_rank,hiring_pressure_score,hiring_pressure_tier,hiring_main_pressure_driver,cross_hiring_rank,cross_hiring_score,...,concurrency_pressure_score,commercial_coverage_pressure_score,sustained_demand_pressure_score,shared_trainer_capacity_pressure_score,workforce_base_score,actions_base_score,commercial_base_score,workforce_vulnerability_tier,actions_pressure_tier,commercial_pressure_tier
0,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,1,77.23,High,Workforce,5,89.38,...,100.000000,96.666667,78.333333,79.166667,82.0,76.69,68.76,High,High,Upper-middle
1,LOG,Logística e Gestão de Armazém,Logística,Centro 5,2,75.61,High,Workforce,7,87.27,...,84.583333,99.166667,96.666667,71.666667,82.0,70.41,73.26,High,Upper-middle,Upper-middle
2,CYB,Cibersegurança Básica,Informática,Centro 1,3,74.42,Upper-middle,Actions,9,86.42,...,76.666667,95.000000,92.500000,95.833333,69.3,81.18,71.17,Upper-middle,High,Upper-middle
3,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 3,4,71.95,Upper-middle,Actions,1,94.98,...,97.916667,85.833333,99.166667,88.333333,56.0,89.71,68.34,Upper-middle,High,Upper-middle
4,EXC,Excel Aplicado à Gestão,Informática,Centro 4,5,71.38,Upper-middle,Actions,12,84.29,...,76.666667,95.833333,93.333333,66.666667,49.3,88.26,81.82,Lower-middle,High,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,VND,Vendas e Negociação,Comercial,Centro 3,116,28.78,Lower-middle,Workforce,113,15.08,...,23.333333,4.166667,7.500000,26.666667,34.8,27.41,19.52,Lower-middle,Lower-middle,Low
116,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,117,27.51,Lower-middle,Workforce,111,17.94,...,16.250000,33.333333,1.666667,0.833333,52.0,8.98,15.60,Upper-middle,Low,Low
117,ING,Inglês Profissional,Línguas,Centro 5,118,27.11,Lower-middle,Workforce,106,21.58,...,23.333333,15.000000,14.166667,17.500000,29.8,26.95,22.05,Lower-middle,Lower-middle,Low
118,TUR,Atendimento Turístico,Turismo,Centro 3,119,24.62,Low,Workforce,120,4.83,...,0.833333,3.333333,8.333333,10.833333,39.8,12.73,18.03,Lower-middle,Low,Low


In [18]:
priority_df = (
    scores_df
    .assign(
        priority_score=lambda df:
            df[
                [
                    "hiring_pressure_score",
                    "cross_hiring_score"
                ]
            ]
            .mean(axis=1)
            .round(2),

        score_difference=lambda df:
            df["hiring_pressure_score"]
            .sub(df["cross_hiring_score"])
            .abs()
            .round(2)
    )
)

In [19]:
priority_df["priority_level"] = pd.cut(
    priority_df["priority_score"],
    bins=[0, 25, 50, 75, 100],
    labels=[
        "Low",
        "Lower-middle",
        "Upper-middle",
        "High"
    ],
    include_lowest=True
)

difference_mean = priority_df["score_difference"].mean()
difference_std = priority_df["score_difference"].std()

agreement_limits = {
    "strong": max(0, difference_mean - difference_std),
    "moderate": difference_mean + difference_std
}

priority_df["score_agreement"] = pd.cut(
    priority_df["score_difference"],
    bins=[
        -float("inf"),
        agreement_limits["strong"],
        agreement_limits["moderate"],
        float("inf")
    ],
    labels=[
        "Strong agreement",
        "Moderate agreement",
        "High disagreement"
    ]
)

In [20]:
priority_df["priority_assessment"] = "Low priority"

priority_df.loc[
    priority_df["priority_level"].eq("High")
    & priority_df["score_agreement"].eq("Strong agreement"),
    "priority_assessment"
] = "Confirmed high priority"

priority_df.loc[
    priority_df["priority_level"].eq("High")
    & priority_df["score_agreement"].eq("Moderate agreement"),
    "priority_assessment"
] = "High priority with moderate agreement"

priority_df.loc[
    priority_df["priority_level"].eq("High")
    & priority_df["score_agreement"].eq("High disagreement"),
    "priority_assessment"
] = "High priority to review"

priority_df.loc[
    priority_df["priority_level"].eq("Upper-middle")
    & priority_df["score_agreement"].isin([
        "Strong agreement",
        "Moderate agreement"
    ]),
    "priority_assessment"
] = "Moderate priority"

priority_df.loc[
    priority_df["priority_level"].eq("Upper-middle")
    & priority_df["score_agreement"].eq("High disagreement"),
    "priority_assessment"
] = "Mixed priority signal"

priority_df.loc[
    priority_df["priority_level"].isin([
        "Low",
        "Lower-middle"
    ])
    & priority_df["score_agreement"].eq("High disagreement"),
    "priority_assessment"
] = "Conflicting signal"

In [21]:
priority_df

,course_id,course_name,course_area,local,hiring_pressure_rank,hiring_pressure_score,hiring_pressure_tier,hiring_main_pressure_driver,cross_hiring_rank,cross_hiring_score,...,actions_base_score,commercial_base_score,workforce_vulnerability_tier,actions_pressure_tier,commercial_pressure_tier,priority_score,score_difference,priority_level,score_agreement,priority_assessment
0,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,1,77.23,High,Workforce,5,89.38,...,76.69,68.76,High,High,Upper-middle,83.30,12.15,High,Moderate agreement,High priority with moderate agreement
1,LOG,Logística e Gestão de Armazém,Logística,Centro 5,2,75.61,High,Workforce,7,87.27,...,70.41,73.26,High,Upper-middle,Upper-middle,81.44,11.66,High,Moderate agreement,High priority with moderate agreement
2,CYB,Cibersegurança Básica,Informática,Centro 1,3,74.42,Upper-middle,Actions,9,86.42,...,81.18,71.17,Upper-middle,High,Upper-middle,80.42,12.00,High,Moderate agreement,High priority with moderate agreement
3,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 3,4,71.95,Upper-middle,Actions,1,94.98,...,89.71,68.34,Upper-middle,High,Upper-middle,83.46,23.03,High,High disagreement,High priority to review
4,EXC,Excel Aplicado à Gestão,Informática,Centro 4,5,71.38,Upper-middle,Actions,12,84.29,...,88.26,81.82,Lower-middle,High,High,77.84,12.91,High,Moderate agreement,High priority with moderate agreement
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,VND,Vendas e Negociação,Comercial,Centro 3,116,28.78,Lower-middle,Workforce,113,15.08,...,27.41,19.52,Lower-middle,Lower-middle,Low,21.93,13.70,Low,Moderate agreement,Low priority
116,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,117,27.51,Lower-middle,Workforce,111,17.94,...,8.98,15.60,Upper-middle,Low,Low,22.72,9.57,Low,Moderate agreement,Low priority
117,ING,Inglês Profissional,Línguas,Centro 5,118,27.11,Lower-middle,Workforce,106,21.58,...,26.95,22.05,Lower-middle,Lower-middle,Low,24.34,5.53,Low,Moderate agreement,Low priority
118,TUR,Atendimento Turístico,Turismo,Centro 3,119,24.62,Low,Workforce,120,4.83,...,12.73,18.03,Lower-middle,Low,Low,14.73,19.79,Low,High disagreement,Conflicting signal


In [22]:
priority_df = priority_df[
    [
        "course_id",
        "course_name",
        "course_area",
        "local",
        
        "hiring_pressure_score",
        "hiring_pressure_tier",

        "cross_hiring_score",
        "cross_hiring_tier",

        "priority_score",
        "score_difference",
        "score_agreement",

        "priority_level",
        "priority_assessment"
    ]
].sort_values(by="priority_score", ascending=False, ignore_index=True)

priority_df

,course_id,course_name,course_area,local,hiring_pressure_score,hiring_pressure_tier,cross_hiring_score,cross_hiring_tier,priority_score,score_difference,score_agreement,priority_level,priority_assessment
0,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 3,71.95,Upper-middle,94.98,High,83.46,23.03,High disagreement,High,High priority to review
1,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,77.23,High,89.38,High,83.30,12.15,Moderate agreement,High,High priority with moderate agreement
2,LOG,Logística e Gestão de Armazém,Logística,Centro 5,75.61,High,87.27,High,81.44,11.66,Moderate agreement,High,High priority with moderate agreement
3,PBI,Power BI e Visualização de Dados,Informática,Centro 2,67.81,Upper-middle,93.62,High,80.72,25.81,High disagreement,High,High priority to review
4,PYT,Introdução à Programação em Python,Informática,Centro 2,69.37,Upper-middle,91.69,High,80.53,22.32,High disagreement,High,High priority to review
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,27.51,Lower-middle,17.94,Low,22.72,9.57,Moderate agreement,Low,Low priority
116,VND,Vendas e Negociação,Comercial,Centro 3,28.78,Lower-middle,15.08,Low,21.93,13.70,Moderate agreement,Low,Low priority
117,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,29.55,Lower-middle,13.69,Low,21.62,15.86,Moderate agreement,Low,Low priority
118,SOC,Socorrismo Básico,Saúde e Segurança,Centro 2,23.56,Low,6.02,Low,14.79,17.54,High disagreement,Low,Conflicting signal


In [23]:
priority_df["priority_assessment"].value_counts().reset_index()

,priority_assessment,count
0,Low priority,54
1,Moderate priority,42
2,Conflicting signal,8
3,High priority to review,6
4,High priority with moderate agreement,5
5,Mixed priority signal,5


In [24]:
priority_df["score_agreement"].value_counts().reset_index()

,score_agreement,count
0,Moderate agreement,76
1,Strong agreement,25
2,High disagreement,19


In [25]:
priority_df["priority_level"].value_counts().reset_index()

,priority_level,count
0,Lower-middle,52
1,Upper-middle,47
2,High,11
3,Low,10


In [26]:
priority_df.loc[
    (priority_df["priority_assessment"] == "Confirmed high priority")
    | (priority_df["priority_assessment"] == "High priority with moderate agreement") 
    | (priority_df["priority_assessment"] == "High priority to review")
]

,course_id,course_name,course_area,local,hiring_pressure_score,hiring_pressure_tier,cross_hiring_score,cross_hiring_tier,priority_score,score_difference,score_agreement,priority_level,priority_assessment
0,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 3,71.95,Upper-middle,94.98,High,83.46,23.03,High disagreement,High,High priority to review
1,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,77.23,High,89.38,High,83.30,12.15,Moderate agreement,High,High priority with moderate agreement
2,LOG,Logística e Gestão de Armazém,Logística,Centro 5,75.61,High,87.27,High,81.44,11.66,Moderate agreement,High,High priority with moderate agreement
3,PBI,Power BI e Visualização de Dados,Informática,Centro 2,67.81,Upper-middle,93.62,High,80.72,25.81,High disagreement,High,High priority to review
4,PYT,Introdução à Programação em Python,Informática,Centro 2,69.37,Upper-middle,91.69,High,80.53,22.32,High disagreement,High,High priority to review
5,CYB,Cibersegurança Básica,Informática,Centro 1,74.42,Upper-middle,86.42,High,80.42,12.00,Moderate agreement,High,High priority with moderate agreement
6,MNT,Manutenção Industrial Básica,Manutenção,Centro 1,68.21,Upper-middle,89.35,High,78.78,21.14,High disagreement,High,High priority to review
7,FOR,Formação Pedagógica Inicial de Formadores,Formação,Centro 3,63.87,Upper-middle,92.06,High,77.96,28.19,High disagreement,High,High priority to review
8,EXC,Excel Aplicado à Gestão,Informática,Centro 4,71.38,Upper-middle,84.29,High,77.84,12.91,Moderate agreement,High,High priority with moderate agreement
9,CYB,Cibersegurança Básica,Informática,Centro 2,68.20,Upper-middle,87.10,High,77.65,18.90,High disagreement,High,High priority to review


In [27]:
priority_df = (
    priority_df
    .assign(
        signed_score_difference=lambda df:
            df["cross_hiring_score"]
            .sub(df["hiring_pressure_score"])
            .round(2)
    )
)

In [28]:
priority_df.groupby(
    "priority_level",
    observed=True
).agg(
    cases=("course_id", "size"),

    hiring_score_mean=(
        "hiring_pressure_score",
        "mean"
    ),

    cross_score_mean=(
        "cross_hiring_score",
        "mean"
    ),

    signed_difference_mean=(
        "signed_score_difference",
        "mean"
    )
).round(2)

,cases,hiring_score_mean,cross_score_mean,signed_difference_mean
priority_level,,,,
Low,10,29.71,13.59,-16.12
Lower-middle,52,42.76,35.50,-7.26
Upper-middle,47,58.19,65.67,7.48
High,11,70.55,89.20,18.65


In [29]:
tier_order = {
    "Low": 0,
    "Lower-middle": 1,
    "Upper-middle": 2,
    "High": 3
}

priority_df = (
    priority_df
    .assign(
        hiring_tier_position=lambda df:
            df["hiring_pressure_tier"]
            .map(tier_order)
            .astype(int),

        cross_tier_position=lambda df:
            df["cross_hiring_tier"]
            .map(tier_order)
            .astype(int)
    )
    .assign(
        tier_difference=lambda df:
            df["hiring_tier_position"]
            .sub(df["cross_tier_position"])
            .abs()
    )
)

In [30]:
priority_df["score_agreement"] = (
    priority_df["tier_difference"]
    .map({
        0: "Strong agreement",
        1: "Moderate agreement",
        2: "High disagreement",
        3: "High disagreement"
    })
)

In [31]:
priority_df["priority_assessment"] = "Low priority"

priority_df.loc[
    priority_df["priority_level"].eq("High")
    & priority_df["score_agreement"].eq("Strong agreement"),
    "priority_assessment"
] = "Confirmed high priority"

priority_df.loc[
    priority_df["priority_level"].eq("High")
    & priority_df["score_agreement"].eq("Moderate agreement"),
    "priority_assessment"
] = "High priority with moderate agreement"

priority_df.loc[
    priority_df["priority_level"].eq("High")
    & priority_df["score_agreement"].eq("High disagreement"),
    "priority_assessment"
] = "High priority to review"

priority_df.loc[
    priority_df["priority_level"].eq("Upper-middle")
    & priority_df["score_agreement"].isin([
        "Strong agreement",
        "Moderate agreement"
    ]),
    "priority_assessment"
] = "Moderate priority"

priority_df.loc[
    priority_df["priority_level"].eq("Upper-middle")
    & priority_df["score_agreement"].eq("High disagreement"),
    "priority_assessment"
] = "Mixed priority signal"

priority_df.loc[
    priority_df["priority_level"].isin([
        "Low",
        "Lower-middle"
    ])
    & priority_df["score_agreement"].eq("High disagreement"),
    "priority_assessment"
] = "Conflicting signal"

In [32]:
priority_df["score_agreement"].value_counts().reset_index()

,score_agreement,count
0,Strong agreement,66
1,Moderate agreement,54


In [33]:
priority_df = priority_df[
    [
        "course_id",
        "course_name",
        "course_area",
        "local",

        "hiring_pressure_score",
        "cross_hiring_score",
        "priority_score",

        "hiring_pressure_tier",
        "cross_hiring_tier",
        "priority_level",

        "priority_assessment"
    ]
]

priority_df

,course_id,course_name,course_area,local,hiring_pressure_score,cross_hiring_score,priority_score,hiring_pressure_tier,cross_hiring_tier,priority_level,priority_assessment
0,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 3,71.95,94.98,83.46,Upper-middle,High,High,High priority with moderate agreement
1,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,77.23,89.38,83.30,High,High,High,Confirmed high priority
2,LOG,Logística e Gestão de Armazém,Logística,Centro 5,75.61,87.27,81.44,High,High,High,Confirmed high priority
3,PBI,Power BI e Visualização de Dados,Informática,Centro 2,67.81,93.62,80.72,Upper-middle,High,High,High priority with moderate agreement
4,PYT,Introdução à Programação em Python,Informática,Centro 2,69.37,91.69,80.53,Upper-middle,High,High,High priority with moderate agreement
...,...,...,...,...,...,...,...,...,...,...,...
115,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,27.51,17.94,22.72,Lower-middle,Low,Low,Low priority
116,VND,Vendas e Negociação,Comercial,Centro 3,28.78,15.08,21.93,Lower-middle,Low,Low,Low priority
117,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,29.55,13.69,21.62,Lower-middle,Low,Low,Low priority
118,SOC,Socorrismo Básico,Saúde e Segurança,Centro 2,23.56,6.02,14.79,Low,Low,Low,Low priority


In [34]:
priority_df["priority_assessment"].value_counts().reset_index()

,priority_assessment,count
0,Low priority,62
1,Moderate priority,47
2,High priority with moderate agreement,9
3,Confirmed high priority,2


In [35]:
priority_df["priority_level"].value_counts().reset_index()

,priority_level,count
0,Lower-middle,52
1,Upper-middle,47
2,High,11
3,Low,10


In [36]:
high_priorities = priority_df.loc[
    (priority_df["priority_assessment"] == "Confirmed high priority")
    | (priority_df["priority_assessment"] == "High priority with moderate agreement") 
    | (priority_df["priority_assessment"] == "High priority to review")
]

high_priorities

,course_id,course_name,course_area,local,hiring_pressure_score,cross_hiring_score,priority_score,hiring_pressure_tier,cross_hiring_tier,priority_level,priority_assessment
0,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 3,71.95,94.98,83.46,Upper-middle,High,High,High priority with moderate agreement
1,SST,Saúde e Segurança no Trabalho,Segurança,Centro 1,77.23,89.38,83.30,High,High,High,Confirmed high priority
2,LOG,Logística e Gestão de Armazém,Logística,Centro 5,75.61,87.27,81.44,High,High,High,Confirmed high priority
3,PBI,Power BI e Visualização de Dados,Informática,Centro 2,67.81,93.62,80.72,Upper-middle,High,High,High priority with moderate agreement
4,PYT,Introdução à Programação em Python,Informática,Centro 2,69.37,91.69,80.53,Upper-middle,High,High,High priority with moderate agreement
5,CYB,Cibersegurança Básica,Informática,Centro 1,74.42,86.42,80.42,Upper-middle,High,High,High priority with moderate agreement
6,MNT,Manutenção Industrial Básica,Manutenção,Centro 1,68.21,89.35,78.78,Upper-middle,High,High,High priority with moderate agreement
7,FOR,Formação Pedagógica Inicial de Formadores,Formação,Centro 3,63.87,92.06,77.96,Upper-middle,High,High,High priority with moderate agreement
8,EXC,Excel Aplicado à Gestão,Informática,Centro 4,71.38,84.29,77.84,Upper-middle,High,High,High priority with moderate agreement
9,CYB,Cibersegurança Básica,Informática,Centro 2,68.20,87.10,77.65,Upper-middle,High,High,High priority with moderate agreement


In [37]:
moderates_priorities = priority_df.loc[
    (priority_df["priority_assessment"] == "Moderate priority")
    | (priority_df["priority_assessment"] == "Mixed priority signal") 
]

moderates_priorities

,course_id,course_name,course_area,local,hiring_pressure_score,cross_hiring_score,priority_score,hiring_pressure_tier,cross_hiring_tier,priority_level,priority_assessment
11,EXC,Excel Aplicado à Gestão,Informática,Centro 2,66.14,81.60,73.87,Upper-middle,High,Upper-middle,Moderate priority
12,CYB,Cibersegurança Básica,Informática,Centro 4,67.63,77.98,72.81,Upper-middle,High,Upper-middle,Moderate priority
13,VND,Vendas e Negociação,Comercial,Centro 4,66.30,78.60,72.45,Upper-middle,High,Upper-middle,Moderate priority
14,PBI,Power BI e Visualização de Dados,Informática,Centro 4,59.78,84.96,72.37,Upper-middle,High,Upper-middle,Moderate priority
15,MNT,Manutenção Industrial Básica,Manutenção,Centro 5,66.14,77.65,71.90,Upper-middle,High,Upper-middle,Moderate priority
16,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 5,62.90,78.73,70.82,Upper-middle,High,Upper-middle,Moderate priority
17,PBI,Power BI e Visualização de Dados,Informática,Centro 3,64.12,77.04,70.58,Upper-middle,High,Upper-middle,Moderate priority
18,INF,"Infeção, Prevenção e Controlo",Saúde,Centro 3,65.43,75.67,70.55,Upper-middle,High,Upper-middle,Moderate priority
19,PBI,Power BI e Visualização de Dados,Informática,Centro 5,65.36,75.15,70.26,Upper-middle,High,Upper-middle,Moderate priority
20,GER,Geriatria e Apoio ao Idoso,Saúde,Centro 4,62.09,76.10,69.10,Upper-middle,High,Upper-middle,Moderate priority


In [38]:
low_priorities = priority_df.loc[
    (priority_df["priority_assessment"] == "Low priority")
    | (priority_df["priority_assessment"] == "Conflicting signal") 
]

low_priorities

,course_id,course_name,course_area,local,hiring_pressure_score,cross_hiring_score,priority_score,hiring_pressure_tier,cross_hiring_tier,priority_level,priority_assessment
58,LID,Liderança e Comunicação,Gestão,Centro 5,50.07,49.48,49.78,Upper-middle,Lower-middle,Lower-middle,Low priority
59,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 3,48.98,48.98,48.98,Lower-middle,Lower-middle,Lower-middle,Low priority
60,MNT,Manutenção Industrial Básica,Manutenção,Centro 3,47.16,50.33,48.74,Lower-middle,Upper-middle,Lower-middle,Low priority
61,SEG,Segurança no Trabalho,Segurança,Centro 2,56.34,40.52,48.43,Upper-middle,Lower-middle,Lower-middle,Low priority
62,GES,Gestão de Equipas,Gestão,Centro 2,48.98,46.60,47.79,Lower-middle,Lower-middle,Lower-middle,Low priority
...,...,...,...,...,...,...,...,...,...,...,...
115,SOC,Socorrismo Básico,Saúde e Segurança,Centro 1,27.51,17.94,22.72,Lower-middle,Low,Low,Low priority
116,VND,Vendas e Negociação,Comercial,Centro 3,28.78,15.08,21.93,Lower-middle,Low,Low,Low priority
117,HOT,Housekeeping e Operações Hoteleiras,Turismo,Centro 4,29.55,13.69,21.62,Lower-middle,Low,Low,Low priority
118,SOC,Socorrismo Básico,Saúde e Segurança,Centro 2,23.56,6.02,14.79,Low,Low,Low,Low priority


In [39]:
priority_df.to_csv(
    OUTPUTS_DATA_DIR / "priority.csv",
    index=False,
    encoding="utf-8"
)